In [1]:
import os
from glob import glob
from pathlib import Path
from dataclasses import dataclass
import json


def get_sort_key(path_str):
    path = Path(path_str)
    # Find the 'garments_5000_X' part
    garment_dir = next(part for part in path.parts if part.startswith('garments_5000_'))
    # Extract the number after 'garments_5000_'
    garment_num = int(garment_dir.split('_')[-1])
    # Get the random ID from the path
    random_id = path.parent.name
    return (garment_num, random_id)


@dataclass
class PathConfig :
    root_path : str = None
    avatar_dir: str = "CLO_ASSETS/AVATARs"
    fabric_dir: str = "CLO_ASSETS/FABRICs"
    pose_dir: str = "CLO_ASSETS/POSEs"
    viewpoint_dir: str = "CLO_ASSETS/VIEWPOINTs"
    
    gcd_dir: str = "GarmentCodeData_v2"

    outfit_metadata_path: str = "gcd_01_outfit_path_list.json"
    combination_metadata_path: str = "gcd_01_top_bottom_path_list.json"
    
    sample_data_dir: str = r"gcd_01\GCD_01"
    
    outfit_path_list: list = None
    combination_path_list: list = None

    def __post_init__(self):
        self.avatar_path_list = sorted(glob(os.path.join(self.root_path, self.avatar_dir, "*.avt")))
        self.fabric_path_list = sorted(glob(os.path.join(self.root_path, self.fabric_dir, "*.zfab")))
        self.pose_path_list = sorted(glob(os.path.join(self.root_path, self.pose_dir, "*.pos")))
        self.viewpoint_path_list = sorted(glob(os.path.join(self.root_path, self.viewpoint_dir, "*.zcmr")))
        
        self.gcd_path_list = sorted(
            glob(os.path.join(
                self.root_path, self.gcd_dir,
                "*", "*", "*config.json"
            )),
            key=get_sort_key
        )
        
        self.sample_data_dir = os.path.join(self.root_path, self.sample_data_dir)
        
        self.outfit_metadata_path = os.path.join(self.root_path, self.outfit_metadata_path)
        self.combination_metadata_path = os.path.join(self.root_path, self.combination_metadata_path)

        with open(self.outfit_metadata_path, "r") as f:
            self.outfit_metadata = json.load(f)
        self.outfit_path_list = []
        for outfit in self.outfit_metadata:
            garment_split, _, garment_id = list(Path(outfit).parts)[-3:]
            
            self.outfit_path_list.append(os.path.join(
                self.sample_data_dir, garment_split, garment_id,
                f"{garment_id}__01__clo.json"
            ))

        with open(self.combination_metadata_path, "r") as f:
            self.combination_metadata_raw = json.load(f)
        self.combination_path_list = []
        for combination in self.combination_metadata_raw:
            top_base_path, bottom_base_path = combination.split(",")
            top_garment_split, _, top_garment_id = list(Path(top_base_path).parts)[-3:]
            bottom_garment_split, _, bottom_garment_id = list(Path(bottom_base_path).parts)[-3:]
            self.combination_path_list.append(os.path.join(
                self.sample_data_dir, top_garment_split,
                f"{top_garment_id}__01__{bottom_garment_id}__01",
                f"{top_garment_id}__01__{bottom_garment_id}__01__clo.json"
            ))
            
    @property
    def avatar_count(self) -> int:
        return len(self.avatar_path_list)

    @property
    def fabric_count(self) -> int:
        return len(self.fabric_path_list)
        
    @property
    def pose_count(self) -> int:
        return len(self.pose_path_list)
    
    @property
    def viewpoint_count(self) -> int:
        return len(self.viewpoint_path_list)

    @property
    def gcd_count(self) -> int:
        return len(self.gcd_path_list)
    
SYSTEM_CONFIG_DICT = {
    "HJP_WINDOWS_DESKTOP": {
        "CLO_DIR": "E:/HJP/KUAICV/VTO/DATA/CLO",
    }
}

In [46]:

system_name = "HJP_WINDOWS_DESKTOP"

path_config = PathConfig(root_path=SYSTEM_CONFIG_DICT[system_name]["CLO_DIR"])

garment_path_list = path_config.combination_path_list
# garment_path_list = path_config.outfit_path_list


In [16]:

print(Path(path_config.pose_path_list[0]))
print(Path(path_config.pose_path_list[0]).stem)
print(type(Path(path_config.pose_path_list[0]).stem))

E:\HJP\KUAICV\VTO\DATA\CLO\CLO_ASSETS\POSEs\Dramatic.pos
Dramatic
<class 'str'>


In [47]:
import shutil

for garment_path in garment_path_list[:10] :
    garment_base_path = Path(garment_path).parent
    print(garment_base_path)
    print(garment_base_path.name)
    
    shutil.copytree(
        garment_base_path,
        Path("../sample_data", garment_base_path.name),
        dirs_exist_ok=True
    )

E:\HJP\KUAICV\VTO\DATA\CLO\gcd_01\GCD_01\garments_5000_0\rand_WUKTRJLRW1__01__rand_IHD87BSA73__01
rand_WUKTRJLRW1__01__rand_IHD87BSA73__01
E:\HJP\KUAICV\VTO\DATA\CLO\gcd_01\GCD_01\garments_5000_0\rand_I0P17DUK2Z__01__rand_2YV0SWYVC9__01
rand_I0P17DUK2Z__01__rand_2YV0SWYVC9__01
E:\HJP\KUAICV\VTO\DATA\CLO\gcd_01\GCD_01\garments_5000_0\rand_EIL756SI93__01__rand_COMEHXS0VU__01
rand_EIL756SI93__01__rand_COMEHXS0VU__01
E:\HJP\KUAICV\VTO\DATA\CLO\gcd_01\GCD_01\garments_5000_0\rand_FG6DJ6CRVH__01__rand_DYTP2CFT4F__01
rand_FG6DJ6CRVH__01__rand_DYTP2CFT4F__01
E:\HJP\KUAICV\VTO\DATA\CLO\gcd_01\GCD_01\garments_5000_0\rand_NZJOFD6030__01__rand_7QI5T2VU4S__01
rand_NZJOFD6030__01__rand_7QI5T2VU4S__01
E:\HJP\KUAICV\VTO\DATA\CLO\gcd_01\GCD_01\garments_5000_0\rand_3Q1TNL7ED8__01__rand_OBDLX7B1ZM__01
rand_3Q1TNL7ED8__01__rand_OBDLX7B1ZM__01
E:\HJP\KUAICV\VTO\DATA\CLO\gcd_01\GCD_01\garments_5000_0\rand_S7NS96YR9I__01__rand_AHNE5VS939__01
rand_S7NS96YR9I__01__rand_AHNE5VS939__01
E:\HJP\KUAICV\VTO\DATA\CLO\

In [17]:



os.listdir(
    Path(garment_path_list[0]).parent
)

['Custom_View_10_1.png',
 'Custom_View_1_1.png',
 'Custom_View_2_1.png',
 'Custom_View_3_1.png',
 'Custom_View_4_1.png',
 'Custom_View_5_1.png',
 'Custom_View_6_1.png',
 'Custom_View_7_1.png',
 'Custom_View_8_1.png',
 'Custom_View_9_1.png',
 'Cut _ Sew Knit_Pique_3_ALPHA.png',
 'Cut _ Sew Knit_Pique_3_BASE.png',
 'Cut _ Sew Knit_Pique_3_DISP.png',
 'Cut _ Sew Knit_Pique_3_MTL.png',
 'Cut _ Sew Knit_Pique_3_NRM.png',
 'Cut _ Sew Knit_Pique_3_ROUGH.png',
 'Dramatic.mtl',
 'Dramatic.obj',
 'Dramatic_meta_data.xml',
 'Dramatic__Custom_View_10_1.png',
 'Dramatic__Custom_View_1_1.png',
 'Dramatic__Custom_View_6_1.png',
 'Dynamic_2.mtl',
 'Dynamic_2.obj',
 'Dynamic_2_meta_data.xml',
 'Dynamic_2__Custom_View_2_1.png',
 'Dynamic_2__Custom_View_9_1.png',
 'Dynamic__Custom_View_7_1.png',
 'FV2_05__Custom_View_8_1.png',
 'FV2_Mia_01_arm_161866_1246066_2474.jpg',
 'FV2_Mia_01_arm_normal_161868_1246068_2476.jpg',
 'FV2_Mia_01_arm_roughness_161870_1246070_2478.jpg',
 'FV2_Mia_01_body_Covered_161858_1

In [18]:
import trimesh
from analysis_utils import visualize_meshes_plotly

@dataclass
class SeamDressScene :
    base_path: str = None
    
    pose_list: list = None
    view_list: list = None

    mesh_list: list = None

    def __post_init__(self):
        self.pose_list = list(map(
            lambda x: Path(x).stem,
            sorted(glob(os.path.join(Path(self.base_path), "*.obj")))
        ))
        
        self.mesh_list = list(map(
            lambda x: trimesh.load(x),
            sorted(glob(os.path.join(Path(self.base_path), "*.obj")))
        ))


scene = SeamDressScene(base_path=Path(garment_path_list[0]).parent)

scene.pose_list


['Dramatic', 'Dynamic_2']

In [26]:
trimesh_scene = scene.mesh_list[-1]
mesh_list = []
for geometry in trimesh_scene.geometry.values() :
    if isinstance(geometry, trimesh.Trimesh):
        mesh_list.append(geometry)

In [40]:
from analysis_utils import visualize_meshes_plotly

fig = visualize_meshes_plotly(
    mesh_list,
    show_edges=False
)